# Cropping

In [ ]:
from tifffile import TiffFile
import spatialdata as sd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import colorsys
from shapely.geometry import box
import geopandas as gpd
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
from PIL import Image, ImageEnhance
from matplotlib.transforms import Affine2D
from Xenium_5K_IPF_constants import MAJOR_CLUSTER_COLORS, SUBTYPE_HATCHES, SAMPLES_SYMBOLS, POINTS_DICT, WORK_PATH

# Option
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

# Constant Declaration
marg_crop = {
    "FR_s1": [225, 300, 5805, 4760],
    "HC_s1": [300, 160, 6540, 6290],
    "AF_s2": [80, 220, 6820, 5570],
    "FR_s3": [250, 310, 8450, 6275],
    "HC_s3": [420, 100, 7500, 6200],
    "FR_s4": [350, 10, 6950, 5710],
}

In [ ]:
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
subtype_colors = {sb: MAJOR_CLUSTER_COLORS[celltype_df.loc[celltype_df["subtype"]==sb, "cell_type_major"].iloc[0]] for sb in list(SUBTYPE_HATCHES)}

In [ ]:
def similarity_from_2points_ydown(he_pts, gdf_pts):
    he = np.asarray(he_pts, dtype=float)
    gdf = np.asarray(gdf_pts, dtype=float)
    
    p1, p2 = he[0], he[1]
    q1, q2 = gdf[0], gdf[1]

    v = p2 - p1
    w = q2 - q1
    
    nv = np.linalg.norm(v)
    nw = np.linalg.norm(w)

    s = nw / nv  # scale

    ang_v = np.arctan2(v[1], v[0])
    ang_w = np.arctan2(w[1], w[0])
    theta = ang_w - ang_v  # rotation

    c, sn = np.cos(theta), np.sin(theta)
    R = np.array([[c, -sn],
                  [sn,  c]])

    t = q1 - (s * (R @ p1))  # translation

    aff = Affine2D().scale(s).rotate(theta).translate(t[0], t[1])
    info = {"scale": float(s), "theta_deg": float(np.degrees(theta)), "t": (float(t[0]), float(t[1]))}
    return aff, info



In [ ]:
def crop_gdf_by_roi(gdf, xmin, ymin, width, height):

    xmax = xmin + width
    ymax = ymin + height
    roi = box(xmin, ymin, xmax, ymax)

    g = gdf[gdf.intersects(roi)].copy()
    g["geometry"] = g.geometry.clip_by_rect(xmin, ymin, xmax, ymax)
    g = g[~g.geometry.is_empty].copy()
    
    return g, (xmin, xmax, ymin, ymax)

def crop_he_array_for_roi(bright_np, aff, extent, pad_px=50):

    xmin, xmax, ymin, ymax = extent

    inv = aff.inverted()  # GDF -> HE(px)

    corners = np.array([
        [xmin, ymin],
        [xmax, ymin],
        [xmax, ymax],
        [xmin, ymax],
    ], dtype=float)

    he_corners = inv.transform(corners)  # (4,2) in HE px
    xs = he_corners[:, 0]
    ys = he_corners[:, 1]

    H, W = bright_np.shape[0], bright_np.shape[1]

    x0 = int(np.floor(xs.min())) - pad_px
    x1 = int(np.ceil(xs.max())) + pad_px
    y0 = int(np.floor(ys.min())) - pad_px
    y1 = int(np.ceil(ys.max())) + pad_px

    x0 = max(0, x0); y0 = max(0, y0)
    x1 = min(W, x1); y1 = min(H, y1)

    bright_crop = bright_np[y0:y1, x0:x1]

    rgb = bright_np[..., :3].astype(np.float32)
    grey = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    if np.issubdtype(bright_np.dtype, np.integer):
        grey_np = np.clip(np.rint(grey), 0, np.iinfo(bright_np.dtype).max).astype(bright_np.dtype)
    else:
        grey_np = grey.astype(bright_np.dtype) 
    grey_crop = grey_np[y0:y1, x0:x1]

    aff_new = Affine2D().translate(x0, y0) + aff

    return bright_crop, grey_crop, aff_new

In [ ]:
# select sample number

nn = 0
print(SAMPLES_SYMBOLS[nn])

In [ ]:
aff, _ = similarity_from_2points_ydown(
    POINTS_DICT[SAMPLES_SYMBOLS[nn]]["HE_points"],
    POINTS_DICT[SAMPLES_SYMBOLS[nn]]["GDF_points"]
)

In [ ]:
with TiffFile(f"{WORK_PATH}Tissue images/{SAMPLES_SYMBOLS[nn]}.ome.tif") as tif:
    he_img = tif.asarray()
img_p = Image.fromarray(np.flipud(Image.fromarray(np.rot90(he_img))))
img_p = ImageEnhance.Brightness(img_p).enhance(1.3) 
he_img = np.asarray(img_p)  # shape: (H,W,3)
del img_p

In [ ]:
sp_adata = sd.SpatialData.read(f"{WORK_PATH}SpatialData/{SAMPLES_SYMBOLS[nn]}")
gdf = sp_adata.shapes["cell_boundaries"]
sq_adata = sp_adata.tables["table"]
del sp_adata
df_c = celltype_df[celltype_df.index.str.endswith(str(nn))]
df_c.index = [m[:-1]+"1" for m in df_c.index]

gdf["subtype"] = gdf.index.map(df_c["subtype"].to_dict())

# Color-coded with Annotated Cell Subtype

## Sample Overview

In [ ]:
# ROI
xmin, ymin, width, height = marg_crop[SAMPLES_SYMBOLS[nn]]

# GDF cropping
gdf_crop, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

# HE cropping
he_crop, _, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*width/1000, height/1000))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0)
fig.set_facecolor("white")

for ax in [ax1, ax2]:
    ax.axis("off")
    ax.set_aspect("equal")
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[3], extent[2])  

for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
    sb_df.plot(
        ax=ax2,
        facecolor=subtype_colors.get(sb, "gray"),
        edgecolor="none",
        linewidth=0,

    )

ax1.imshow(
    he_crop,
    origin="upper",    
    transform=aff_crop + ax1.transData,
    interpolation="nearest",
)

scale_bar = 1000
x_pad = width*0.03
y_pad = height*0.97
if SAMPLES_SYMBOLS[nn] == "FR_s1":
    ax1.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=4, colors="white")
else:
    ax1.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=4, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_HE_GDF_spatial_{SAMPLES_SYMBOLS[nn]}.png", dpi=300, format="png", bbox_inches=None)
#
plt.show()


## Cropped Images (HE and GDF)

In [ ]:
crop_dict = {
    "00": [[3160, 3820, 440, 280], True, 50, "FF", 300], # FR_s3
    "01": [[3300, 850, 440, 280], True, 50, "Near_normal alveolar", 300], # AF_s2
    "02": [[2980, 2090, 440, 280], True, 50, "Metaplasia", 300], # HC_s1
    "03": [[4160, 500, 900, 1400], False, 200, "Bronchiolar", 100], # FR_s1
    "04": [[4000, 3000, 1700, 2000], False, 500, "Macro_TLS", 100], # AF_s2
}

# select cropping point
point = "00"

In [ ]:
# ROI
xmin, ymin, width, height = crop_dict[point][0]

# GDF cropping
gdf_crop, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

# HE cropping
he_crop, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

# plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12*height/width))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0)
fig.set_facecolor("white")

for ax in [ax1, ax2]:
    ax.axis("off")
    ax.set_aspect("equal")
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[3], extent[2]) 

lw = (-2.3/1e4)*max(crop_dict[point][0][2],crop_dict[point][0][3])+0.7

if crop_dict[point][1]:
    for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
        sb_df.plot(
            ax=ax2,
            facecolor=subtype_colors.get(sb, "gray"),
            edgecolor="#3E3E3E",
            linewidth=lw,
            hatch=SUBTYPE_HATCHES.get(sb, None),
        )
else:
    for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
        sb_df.plot(
            ax=ax2,
            facecolor=subtype_colors.get(sb, "gray"),
            edgecolor="#3E3E3E",
            linewidth=lw,
        )

ax1.imshow(
    he_crop,
    origin="upper",    
    transform=aff_crop + ax1.transData,
    interpolation="nearest"
)

ax2.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax2.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)

scale_bar = crop_dict[point][2]
x_pad = width*0.03
y_pad = height*0.97

ax1.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=10, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_HE_GDF_spatial_crop_{scale_bar}_{SAMPLES_SYMBOLS[nn]}_{crop_dict[point][3]}.png", dpi=crop_dict[point][4], format="png", bbox_inches=None)

plt.show()


## Cropped Images (GDF only)

In [ ]:
# gdf only
crop_dict_gdf = {
    "00": [[4270, 1340, 300, 320], True, 40, "Bronchiolar", 500], # FR_s1 
    "01": [[4250, 3330, 310, 280], True, 40, "Macro1", 500], # AF_s2
    "02": [[5200, 3260, 350, 350], True, 40, "Macro2", 500], # AF_s2 
    "03": [[4770, 4360, 460, 290], True, 40, "TLS", 500], # AF_s2
}

point_gdf = "00"

In [ ]:
# ROI
xmin, ymin, width, height = crop_dict_gdf[point_gdf][0]

# GDF cropping
gdf_crop, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

# HE cropping
he_crop, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

# plot
fig, ax = plt.subplots(figsize=(crop_dict_gdf[point_gdf][0][2]/30, crop_dict_gdf[point_gdf][0][3]/30))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.set_facecolor("white")

ax.axis("off")
ax.set_aspect("equal")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[3], extent[2]) 

if crop_dict_gdf[point_gdf][1]:
    for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
        sb_df.plot(
            ax=ax,
            facecolor=subtype_colors.get(sb, "gray"),
            edgecolor="#3E3E3E",
            linewidth=0.8,
            hatch=SUBTYPE_HATCHES.get(sb, None),
        )
else:
    for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
        sb_df.plot(
            ax=ax,
            facecolor=subtype_colors.get(sb, "gray"),
            edgecolor="#3E3E3E",
            linewidth=0.8,
        )

ax.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)

scale_bar = crop_dict_gdf[point_gdf][2]
x_pad = width*0.03
y_pad = height*0.97
ax.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=crop_dict_gdf[point_gdf][0][3]/30, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_HE_GDF_spatial_crop_{scale_bar}_{SAMPLES_SYMBOLS[nn]}_{crop_dict_gdf[point_gdf][3]}_merge.png", 
            dpi=crop_dict_gdf[point_gdf][4], format="png", bbox_inches=None)

plt.show()


# Color-coded with Epithelial-centered Neighborhood Class

## Sample Overview

In [ ]:
n_adata = sc.read_h5ad(f"{WORK_PATH}AnnData/EcNP.h5ad")

In [ ]:
n_adata_sm = n_adata[n_adata.obs["sample"]==SAMPLES_SYMBOLS[nn]]
n_adata_sm.obs.set_index("cell_id", inplace=True)
sq_adata.obs_names = sq_adata.obs["cell_id"]
sq_adata_ecnp = sq_adata[n_adata_sm.obs.index,].copy()
for hoge in ["EcNP", "leiden"]:
    sq_adata_ecnp.obs[hoge]=n_adata_sm.obs[hoge]
    sq_adata_ecnp.uns[f"{hoge}_colors"] = n_adata_sm.uns[f"{hoge}_colors"]

In [ ]:
# leiden clusters of epithelial-centered neighborhood profiles

# ROI
xmin, ymin, width, height = marg_crop[SAMPLES_SYMBOLS[nn]]

# GDF cropping
_, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

# HE cropping
_, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

# plot
fig, ax = plt.subplots(1, 1, figsize=(1*width/1000, height/1000))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.set_facecolor("white")

ax.axis("off")
ax.set_aspect("equal")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3]) 

sq.pl.spatial_scatter(sq_adata_ecnp, color="leiden", shape=None, library_id="spatial", ax=ax, size=width/350, legend_loc="none")

ax.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)


scale_bar = 1000
x_pad = width*0.03
y_pad = height*0.97
ax.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=6, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_EcNP_spatial_{SAMPLES_SYMBOLS[nn]}_overview_leiden.png", dpi=300, format="png", bbox_inches=None)

plt.show()


In [ ]:
# neighborhood classes

# ROI
xmin, ymin, width, height = marg_crop[SAMPLES_SYMBOLS[nn]]

# GDF cropping
_, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

# HE cropping
_, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

# plot
fig, ax = plt.subplots(1, 1, figsize=(1*width/1000, height/1000))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.set_facecolor("white")

ax.axis("off")
ax.set_aspect("equal")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3]) 

sq.pl.spatial_scatter(sq_adata_ecnp, color="EcNP", shape=None, library_id="spatial", ax=ax, size=width/350)

ax.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)


scale_bar = 1000
x_pad = width*0.03
y_pad = height*0.97
ax.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=6, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_EcNP_spatial_{SAMPLES_SYMBOLS[nn]}_overview.png", dpi=300, format="png", bbox_inches=None)

plt.show()


## Cropped images (GDF only)

In [ ]:
n_df = n_adata_sm.obs
n_col_list= list(n_adata.uns["EcNP_colors"])
n_colors = {le: n_col_list[n] for n, le in enumerate(n_adata.obs["EcNP"].cat.categories)}

In [ ]:
gdf = gdf[~gdf["subtype"].isna()].copy()
gdf["EcNP"] = gdf.index.map(n_df["EcNP"].to_dict())
gdf["EcNP"] = np.where(gdf["EcNP"].isna(), "not epithelial", gdf["EcNP"])
n_colors["not epithelial"] = "#DDDDDD"

In [ ]:
crop_dict = {
    "1": [[3300, 850, 440, 280], True, 50, "Near_normal alveolar"], # AF_s2
    "2": [[3160, 3820, 440, 280], True, 50, "Fibrotic"], # FR_s3
    "3": [[2980, 2090, 440, 280], False, 50, "Metaplasia"], # HC_s1
    "4": [[3920, 1510, 300, 250], False, 50, "Alveolar duct"],  # FR_s1
    "5": [[4160, 500, 900, 1400], False, 200, "Bronchi_duct"],  # FR_s1
    "6": [[3300, 4250, 300, 230], False, 50, "Epi_rich"],  # HC_s1
}

point = "1"

In [ ]:
# only GDF

xmin, ymin, width, height = crop_dict[point][0]

gdf_crop, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

_, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

fig, ax = plt.subplots(1, 1, figsize=(6*width/250, 6*width/250*height/width))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0)
fig.set_facecolor("white")

ax.axis("off")
ax.set_aspect("equal")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[3], extent[2]) 

for sb, sb_df in gdf_crop.groupby("EcNP", sort=False):
    sb_df.plot(
        ax=ax,
        facecolor=n_colors.get(sb, "white"),
        edgecolor="#3E3E3E",
        linewidth=0.5,
    )

ax.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)

scale_bar = crop_dict[point][2]
x_pad = width*0.03
y_pad = height*0.97

ax.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=scale_bar/5, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_EcNP_spatial_{SAMPLES_SYMBOLS[nn]}_{crop_dict[point][3]}_{scale_bar}.png", dpi=300, format="png", bbox_inches=None)

plt.show()

In [ ]:
# panel for HE, cell subtype and neighohood class

xmin, ymin, width, height = crop_dict[point][0]

gdf_crop, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

he_crop, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(3*6*width/250, 6*width/250*height/width))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0)
fig.set_facecolor("white")

for ax in [ax1, ax2, ax3]:
    ax.axis("off")
    ax.set_aspect("equal")
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[3], extent[2])

if crop_dict[point][1]:
    for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
        sb_df.plot(
            ax=ax2,
            facecolor=subtype_colors.get(sb, "gray"),
            edgecolor="#3E3E3E",
            linewidth=0.5,
            hatch=SUBTYPE_HATCHES.get(sb, None),
        )
else:
    for sb, sb_df in gdf_crop.groupby("subtype", sort=False):
        sb_df.plot(
            ax=ax2,
            facecolor=subtype_colors.get(sb, "gray"),
            edgecolor="#3E3E3E",
            linewidth=0.5,
            hatch=SUBTYPE_HATCHES.get(sb, None),
        )


for sb, sb_df in gdf_crop.groupby("EcNP", sort=False):
    sb_df.plot(
        ax=ax3,
        facecolor=n_colors.get(sb, "white"),
        edgecolor="#3E3E3E",
        linewidth=0.5,
    )

ax1.imshow(
    he_crop,
    origin="upper",    
    transform=aff_crop + ax1.transData,
    interpolation="nearest"
)

ax2.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax2.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)

scale_bar = crop_dict[point][2]
x_pad = width*0.03
y_pad = height*0.97

ax1.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=10, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_EcNP_spatial_{SAMPLES_SYMBOLS[nn]}_{crop_dict[point][3]}_{scale_bar}_ax3.png", dpi=300, format="png", bbox_inches=None)

plt.show()

# Color-coded with Pseudo-temporal Continuum Position

## Sample Overview

In [ ]:
ptc_df = pd.read_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index_col="cell_id")

In [ ]:
ptc_df_sm = ptc_df[ptc_df.index.str.endswith(str(nn))]
ptc_df_sm.index = [idx[:-1]+str("1") for idx in ptc_df_sm.index]

In [ ]:
sq_adata.obs_names = sq_adata.obs["cell_id"]
sq_adata_ptc = sq_adata[ptc_df_sm.index,].copy()
sq_adata_ptc.obs["C_position"]=ptc_df_sm["C_position"]

In [ ]:
xmin, ymin, width, height = marg_crop[SAMPLES_SYMBOLS[nn]]

_, extent = crop_gdf_by_roi(gdf, xmin, ymin, width, height)

_, gr_crop, aff_crop = crop_he_array_for_roi(he_img, aff, extent, pad_px=80)

fig, ax = plt.subplots(1, 1, figsize=(1*width/1000, height/1000))
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
fig.set_facecolor("white")

ax.axis("off")
ax.set_aspect("equal")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3]) 

sq.pl.spatial_scatter(sq_adata_ptc, color="C_position", shape=None, library_id="spatial", ax=ax, size=width/350, colorbar=False, cmap="plasma")

ax.imshow(
    gr_crop,
    origin="upper",    
    transform=aff_crop + ax.transData,
    interpolation="nearest",
    cmap="grey",
    alpha=0.5
)


scale_bar = 1000
x_pad = width*0.03
y_pad = height*0.97
ax.hlines(y=ymin+y_pad, xmin=xmin+x_pad, xmax=scale_bar+xmin+x_pad, linewidth=6, colors="black")
fig.savefig(f"{WORK_PATH}Figure/041_PtC_spatial_{SAMPLES_SYMBOLS[nn]}_overview.png", dpi=300, format="png", bbox_inches=None)

plt.show()
